In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.datasets as datasets
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
from torch.optim.lr_scheduler import StepLR
from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter

from typing import Dict, Tuple
from tqdm import tqdm
import numpy as np
import time
import os
import random
from tabulate import tabulate

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib inline

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")



In [ ]:
from waveguide_dataset_paired import WaveguideDatasetPaired
dataset = WaveguideDatasetPaired('train_test_split.h5')

In [ ]:
import torch
import torch.nn as nn

class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

class ResidualBlockGN(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.gn1   = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.gelu1 = nn.GELU()

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.gn2   = nn.GroupNorm(num_groups=8, num_channels=out_channels)

        self.downsample = None
        if downsample or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )

        self.activation = nn.GELU()

    def forward(self, x):
        identity = x

        out = self.gelu1(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))

        if self.downsample:
            identity = self.downsample(identity)

        out += identity
        return self.activation(out)

class Net4_Mode0Weight0(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: [B, 1, 32, 32]
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(8, 64),
            nn.GELU()
        )

        # ---------- Residual Blocks (10 conv layers total = 5 residual blocks) ----------
        self.res_blocks = nn.Sequential(
            ResidualBlockGN(64, 64),                # Conv 1–2
            ResidualBlockGN(64, 128, downsample=True),  # Conv 3–4, [B, 128, 16, 16]
            nn.Dropout(0.25),

            ResidualBlockGN(128, 256, downsample=True), # Conv 5–6, [B, 256, 8, 8]
            nn.Dropout(0.25),

            ResidualBlockGN(256, 512, downsample=True), # Conv 7–8, [B, 512, 4, 4]
            nn.Dropout(0.25),

            ResidualBlockGN(512, 512),              # Conv 9–10
        )

        self.flatten = Flatten()  # Output shape: [B, 512*4*4] = [B, 8192]

        # ---------- Fully Connected Head ----------
        def groupnorm_1d(features, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=features)

        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048),
            groupnorm_1d(2048), nn.GELU(), nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            groupnorm_1d(1024), nn.GELU(), nn.Dropout(0.3),

            nn.Linear(1024, 256),
            groupnorm_1d(256), nn.GELU(), nn.Dropout(0.25),

            nn.Linear(256, 1)
        )

    def forward(self, x_img, x_cond):
        print("x_cond shape in forward:", x_cond.shape)
        x = self.stem(x_img)           # [B, 64, 32, 32]
        x = self.res_blocks(x)         # [B, 512, 4, 4]
        x = self.flatten(x)            # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)  # [B, 8196]
        return self.fc(x)              # [B, 1]


In [ ]:
def train_structured(model, device, loader, optimizer, loss_fn):
    """
    Train for one epoch on structured [4,2] mode-weight inputs.

    · model takes:   waveguide, params
    · model outputs: [mode0, weight0]
    · target:        cond[0] (mode0, weight0), extracted from the [4,2] tensor
    """
    model.train()

    for cond, params, waveguide in loader:
        cond, params, waveguide = (
            cond.to(device),       # [B, 4, 2]
            params.to(device),     # [B, 4]
            waveguide.to(device),  # [B, 1, 32, 32]
        )

        # ✂ Extract mode0 and weight0 as target from cond[:, 0]
        y_true = cond[:, 0, 1].unsqueeze(1)  # shape [B, 1]

        # forward pass
        optimizer.zero_grad()
        y_pred = model(waveguide, params)  # shape [B, 2]
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()

In [ ]:
def test_structured(model, device, loader, loss_fn, dataset, epoch_num, total_epochs):
    """
    Evaluate model for one epoch using [B, 4, 2] structured mode-weight targets.

    * Model outputs:   [B, 1] (weight-0 only)
    * Ground truth:    [B, 4, 2]; we slice [:, 0, 1] for the first weight

    Returns
    -------
    float
        Average MSE loss over the test set.
    """
    model.eval()
    total_loss = 0.0
    collected  = []

    # scalar stats for inverse normalization of weight0
    wt_mean_log = float(dataset.meanw[0])
    wt_std_log  = float(dataset.stdw[0])

    with torch.no_grad():
        for cond, params, waveguide in loader:
            cond, params, waveguide = (
                cond.to(device),       # [B, 4, 2]
                params.to(device),     # [B, 4]
                waveguide.to(device),  # [B, 1, 32, 32]
            )

            # Extract only weight0 target: shape [B, 1]
            y_true = cond[:, 0, 1].unsqueeze(1)

            # Predict: shape [B, 1]
            y_pred = model(waveguide, params)

            batch_loss = loss_fn(y_pred, y_true).item()
            total_loss += batch_loss * waveguide.size(0)

            # Collect samples for plotting
            for t, o, p in zip(y_true.cpu(), y_pred.cpu(), params.cpu()):
                if len(collected) < 50:
                    t_weight0_log = t.item() * wt_std_log + wt_mean_log
                    o_weight0_log = o.item() * wt_std_log + wt_mean_log
                    t_weight0 = np.expm1(t_weight0_log)
                    o_weight0 = np.expm1(o_weight0_log)

                    collected.append((
                        t_weight0,
                        o_weight0,
                        p.numpy()
                    ))

    avg_loss = total_loss / len(loader.dataset)

    # Plot on last epoch
    if epoch_num == total_epochs - 1 and collected:
        chosen = random.sample(collected, 8)
        fig, ax = plt.subplots(figsize=(6, 4))

        for tgt, out, prm in chosen:
            ax.plot([0], [tgt], 'ro')   # target
            ax.plot([0], [out], 'bx')   # output

        ax.set_title("Weight-0 Prediction")
        ax.set_xticks([0])
        ax.set_xticklabels(['value'])
        ax.grid(True)

        plt.suptitle("Targets (red) vs Outputs (blue) on last epoch")
        plt.tight_layout()
        plt.show()

    return avg_loss


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import torch.nn as nn

def main(dataset):
    os.makedirs("models", exist_ok=True)
    
    # --- Training Configuration ---
    batch_size = 512
    test_batch_size = 1000
    lr = 1e-3
    gamma = 0.9
    epochs = 200
    save_dir = 'only_first_weight_no_mode'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- Model & Optimizer ---
    model = Net4_Mode0Weight0().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.MSELoss()

    # --- Dataset Split ---
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=8)

    # --- Training Loop ---
    e_loss_graph = []
    pbar = tqdm(range(epochs), desc="Training, loss= ----")

    for epoch in pbar:
        train_structured(model, device, train_loader, optimizer, loss_fn)
        e_loss = test_structured(model, device, test_loader, loss_fn, dataset, epoch, epochs)
        e_loss_graph.append(e_loss)

        # Update progress bar description
        pbar.set_description(f"Training, loss: {e_loss:.4f}")

        # Scheduler step and save model
        scheduler.step()
        torch.save(model.state_dict(), f"models/{save_dir}.pth")

        # Save updated loss plot
        plt.figure()
        plt.plot(range(epoch + 1), e_loss_graph, label="Test Loss")
        plt.title(f"Test Loss Over Epochs ({save_dir})")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(True)
        plt.legend()
        plt.savefig(f"models/{save_dir}.png")
        plt.close()

if __name__ == '__main__':
    main(dataset)
